# Medical Device Safety Intelligence: bounded data exploration

This notebook explores the validated local CSV snapshot of openFDA MAUDE reports and device recalls. It checks table integrity and summarizes event types, device entries, product-problem entries and the small recall extract. The selected window is **September 1–October 13, 2026**; this short extract is not a long-term trend dataset.

Sources: [openFDA device event API](https://open.fda.gov/apis/device/event/) and [openFDA device recall API](https://open.fda.gov/apis/device/recall/). Run the bounded preparation command described in the [README](../README.md) to create `data/processed/` before running this notebook.

MAUDE contains reports of suspected device-associated events. Counts do not establish causality, incidence, failure rates or comparative safety. Recall records describe a separate regulatory process and are not linked to individual MAUDE reports here.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Resolve files from either the repository root or this notebook's folder.
candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((p for p in candidates if (p / 'data' / 'processed').is_dir()), None)
if project_root is None:
    raise FileNotFoundError('Could not find data/processed. Run the preparation command from the README first.')
data_dir = project_root / 'data' / 'processed'
files = {
    'reports': 'maude_reports.csv',
    'devices': 'maude_devices.csv',
    'problems': 'maude_product_problems.csv',
    'recalls': 'device_recalls.csv',
    'quality': 'data_quality_summary.csv',
}
missing = [name for name in files.values() if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing processed CSV files: {missing}')
tables = {key: pd.read_csv(data_dir / name, dtype='string') for key, name in files.items()}
reports, devices, problems, recalls, quality = (tables[k] for k in ('reports', 'devices', 'problems', 'recalls', 'quality'))
for frame, columns in [(reports, ['event_date', 'date_received']), (recalls, ['event_date_initiated', 'event_date_posted', 'termination_date'])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column], errors='coerce')
print(f'Project: {project_root}')
print('Loaded rows:', {name: len(frame) for name, frame in tables.items()})

## Data quality and relationships

The child tables can contain multiple rows per report. Their counts describe entries, not unique incidents. The checks below compare keys and dates within this selected extract.

In [ ]:
report_key = 'mdr_report_key'
report_keys = set(reports[report_key].dropna())
def key_summary(frame, key_columns):
    missing_rows = frame[key_columns].isna().any(axis=1).sum()
    complete = frame.dropna(subset=key_columns)
    return complete.drop_duplicates(key_columns).shape[0], missing_rows, complete.duplicated(key_columns).sum()
report_key_stats = key_summary(reports, [report_key])
device_key_stats = key_summary(devices, [report_key, 'device_ordinal'])
problem_key_stats = key_summary(problems, [report_key, 'problem_ordinal'])
recall_key_stats = key_summary(recalls, ['cfres_id'])
checks = pd.DataFrame([
    ('MAUDE reports', len(reports), *report_key_stats, None),
    ('Device entries', len(devices), *device_key_stats, (~devices[report_key].isin(report_keys)).sum()),
    ('Product-problem entries', len(problems), *problem_key_stats, (~problems[report_key].isin(report_keys)).sum()),
    ('Recall records', len(recalls), *recall_key_stats, None),
], columns=['Table', 'Rows', 'Distinct primary keys', 'Rows missing key fields', 'Duplicate primary-key rows', 'Orphan report keys'])
checks

In [ ]:
date_quality = pd.DataFrame([
    {'Table': 'MAUDE event date', 'Nonmissing': reports['event_date'].notna().sum(), 'Start': reports['event_date'].min(), 'End': reports['event_date'].max()},
    {'Table': 'MAUDE received date', 'Nonmissing': reports['date_received'].notna().sum(), 'Start': reports['date_received'].min(), 'End': reports['date_received'].max()},
    {'Table': 'Recall initiation date', 'Nonmissing': recalls['event_date_initiated'].notna().sum(), 'Start': recalls['event_date_initiated'].min(), 'End': recalls['event_date_initiated'].max()},
])
display(date_quality)
display(quality[['source', 'selected_start', 'selected_end', 'retrieved_records', 'unique_records', 'duplicate_records', 'missing_date_records_in_selected_rows', 'invalid_date_records', 'out_of_range_records']])

## Reported event types
This ranks report records by reported event type. It is a description of this selected dataset, not a measure of risk or incidence.

In [ ]:
event_counts = reports['event_type'].fillna('Not reported').value_counts().head(10).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
event_counts.plot.barh(ax=ax, color='#6252a3')
ax.set(title='Most common reported event types', xlabel='MAUDE reports', ylabel='Reported event type')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

## Product-problem and device entries
Problem categories count rows in `maude_product_problems`; one report may have multiple entries. Device counts similarly describe device entries grouped by the reported product code, not unique incidents or device use.

In [ ]:
problem_counts = problems['product_problem'].fillna('Not reported').value_counts().head(12).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
problem_counts.plot.barh(ax=ax, color='#168b83')
ax.set(title='Most frequent product-problem entries', xlabel='Entries (not unique reports)', ylabel='Reported product problem')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
device_counts = devices['device_report_product_code'].fillna('Not reported').value_counts().head(12).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
device_counts.plot.barh(ax=ax, color='#d88945')
ax.set(title='Most frequent device product-code entries', xlabel='Device entries', ylabel='FDA product code')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

## Recall records
Only 11 recall records are in this extract. Treat these as individual records for review; the sample is too small for broad comparisons or a meaningful trend chart. Recall status and initiation-date summaries do not link recalls to individual MAUDE reports.

In [ ]:
recall_summary = recalls.groupby('recall_status', dropna=False).size().rename('Recall records').reset_index()
display(recall_summary)
display(recalls[['cfres_id', 'event_date_initiated', 'recall_status', 'product_code']].sort_values('event_date_initiated'))

## Interpretation
Use the figures as a descriptive view of a bounded extract. MAUDE reports are not verified causal events, and counts lack the exposure denominators needed for incidence or comparative safety conclusions. The recall table represents a separate process; product-code overlap alone does not establish that a recall caused or explains an individual report. Re-run the notebook after preparing a new validated snapshot, and check its date window and quality summary before comparing results.